In [1]:
import numpy as np
import pandas as pd

In [2]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

In [3]:
import re

def text_clean(text):
  lower_text = text.lower()
  simple_text = re.sub(r'[^a-z0-9\s]', '', lower_text)
  cleaned_text = re.sub(r'\s+', ' ', simple_text)
  return cleaned_text

In [4]:
#applying function to all the columns in both train and test
cols = ['prompt' , 'A', 'B' , 'C' , 'D' , 'E']

for col in cols:
    train[col +'_clean'] = train[col].apply(text_clean)
    test[col +'_clean'] = test[col].apply(text_clean)

In [5]:
train.head(2)

,id,prompt,A,B,C,D,E,answer,prompt_clean,A_clean,B_clean,C_clean,D_clean,E_clean
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,pick the best possible answer what is martin h...,martin heidegger believes that humans exist wi...,martin heidegger believes that humans do not e...,martin heidegger does not believe in the exist...,martin heidegger believes that the relationshi...,martin heidegger believes that time is an illu...
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A,what is acceleratorbased lightion fusion,acceleratorbased lightion fusion is a techniqu...,acceleratorbased lightion fusion is a techniqu...,acceleratorbased lightion fusion is a techniqu...,acceleratorbased lightion fusion is a techniqu...,acceleratorbased lightion fusion is a techniqu...


In [6]:
def map_at3(prediction,correct):  #function to calculate MAP@3 score
  score = 0
  for pred, x in zip(prediction, correct):
    if x in pred:
      k = pred.index(x) + 1
      score += 1/k
  return round(score/len(correct) ,4)

In [7]:
x = train.copy()

In [10]:
from sklearn.model_selection import train_test_split

x_train, x_test = train_test_split(x, test_size=0.15, random_state=42 , stratify = x['answer'])

In [15]:
x_train.shape , x_test.shape


((1700, 14), (300, 14))

In [16]:
#function to build prompt and option pairs

def build_pairs(df):
    rows = []
    for i in df.index:
        prompt = df.loc[i, 'prompt_clean']
        options = ['A', 'B', 'C', 'D', 'E']
        for opt in options:
            text = prompt + ' ' + df.loc[i, f'{opt}_clean']
            label = 0
            if df.loc[i, 'answer'] == opt:
                label = 1

            rows.append({'id': df.loc[i, 'id'],
                         'option': opt,
                         'text': text,
                         'label': label})
    return pd.DataFrame(rows)

train_pairs = build_pairs(x_train)
val_pairs = build_pairs(x_test)

print(train_pairs.shape, val_pairs.shape)

(8500, 4) (1500, 4)


In [17]:
train_pairs.iloc[0:5]

,id,option,text,label
0,1119,A,pick the best possible answer which hand shoul...,0
1,1119,B,pick the best possible answer which hand shoul...,1
2,1119,C,pick the best possible answer which hand shoul...,0
3,1119,D,pick the best possible answer which hand shoul...,0
4,1119,E,pick the best possible answer which hand shoul...,0
